# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Vedika1304-05/flyrank-internship-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
1. What one row means: one row = one content item, for one client, on one specific calendar day — the grain of fact_content_daily_performance is report_date + client_hash_id + content_hash_id. This is a genuine daily time series, unlike the starter CSV's single 90-day snapshot per page.
2. Which table(s): primarily fact_content_daily_performance (the daily measurements), joined to dim_content (content metadata, join key content_hash_id) and dim_clients (for gsc_data_start/ga4_data_start, join key client_hash_id). fact_content_query_90d is optional/secondary, for query-mix features later.
3. Which time window: developing against one mid-panel month, month=2026-03, as per the guide's own advice. This avoids the _sample table (which holds the last month and risks peeking at the future).
4. What I would predict: The future-window proxy - will a page decline or will a page's impressions decline by >=20% (for eg.) in the future, considering the data of the previous 30-days.
5. What I deliberately exclude: trend_pct-equivalent derived trend columns, and any pre-built decision flags — since we already proved on the starter data that a trend-percentage column is what a "declining" bucket is thresholded from, using it as a feature would be leakage.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Features: gsc_impressions, gsc_clicks, gsc_avg_position, GA4 sessions, scroll events (per the density table), content age/freshness signals from dim_content. These are observable signals, known before the decision point — real measurements of what happened, not decisions.

Label: 	A future-window decline flag you construct yourself from gsc_impressions (or similar), comparing a window after the decision point to the window before it. This is the outcome which we're predicting & it must come from measured future window.

Context: 	client_hash_id, content_hash_id, report_date, keyword_hash_id, url_hash_id, access_profile. These are useful for joining, grouping, and interpreting results — but not fed to the model as a predictive signal, since the hashes themselves carry no real information.

Excluded: ny pre-built trend/decision column (e.g. a trend_pct-style derived column, if the warehouse has one), plus gsc_data_start/ga4_data_start used as features. This is beacuse of the rule against circularity/leakage — a signal that already encodes or overlaps with the label shouldn't also predict it. Including these features will cause the model to overfit the data and learn the specific relations instead of general patterns, due to which it will perform poorly on test data.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# Mid-panel month used: 2026-03 (avoids the full table + avoids the sample table)
# ============================================================

MONTH_START = "2026-03-01"
MONTH_END   = "2026-04-01"  # exclusive upper bound

FACT_MONTH = f"""
    SELECT *
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date < DATE '{MONTH_END}'
"""

# ------------------------------------------------------------
# FACT 1 — Grain: one row = one client x one content item x one day
# ------------------------------------------------------------
grain_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (client_hash_id, content_hash_id, report_date)) AS unique_combinations
    FROM ({FACT_MONTH})
""").df()

print("FACT 1 — Grain check")
print(grain_check.to_string(index=False))
print(f"Grain confirmed: {grain_check['total_rows'][0] == grain_check['unique_combinations'][0]}\n")

# ------------------------------------------------------------
# FACT 2 — Row count and date span for this slice
# ------------------------------------------------------------
span_check = con.sql(f"""
    SELECT
        COUNT(*) AS n_rows,
        MIN(report_date) AS earliest_date,
        MAX(report_date) AS latest_date,
        COUNT(DISTINCT client_hash_id) AS n_clients,
        COUNT(DISTINCT content_hash_id) AS n_content_items
    FROM ({FACT_MONTH})
""").df()

print("FACT 2 — Row count and date span")
print(span_check.to_string(index=False))
print()

# ------------------------------------------------------------
# FACT 3 — Availability, filtered with IS TRUE
# ------------------------------------------------------------
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS rows_before_filter,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_surviving,
        ROUND(
            SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1
        ) AS pct_surviving
    FROM ({FACT_MONTH})
""").df()

print("FACT 3 — Availability filter (IS TRUE)")
print(availability_check.to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FACT 1 — Grain check
 total_rows  unique_combinations
    9841378              9841378
Grain confirmed: True



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FACT 2 — Row count and date span
 n_rows earliest_date latest_date  n_clients  n_content_items
9841378    2026-03-01  2026-03-31         55           331437

FACT 3 — Availability filter (IS TRUE)
 rows_before_filter  rows_surviving  pct_surviving
            9841378        413966.0            4.2


In [19]:
# ============================================================
# Five features, max, from the March 2026 slice
# Decision point = 2026-03-31 ("pretend today" for this build)
# Every feature uses ONLY rows dated on or before that point.
# ============================================================

MONTH_START     = "2026-03-01"
DECISION_POINT  = "2026-03-31"

feature_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS total_impressions_30d,
        SUM(gsc_clicks) AS total_clicks_30d,
        ROUND(SUM(gsc_clicks) * 1.0 / NULLIF(SUM(gsc_impressions), 0), 4) AS ctr_30d,
        ROUND(AVG(gsc_avg_position), 2) AS avg_position_30d,
        COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impressions_30d
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}'
      AND report_date <= DATE '{DECISION_POINT}'
    GROUP BY client_hash_id, content_hash_id
    HAVING SUM(gsc_impressions) > 0
""").df()

print(f"Feature frame: {feature_frame.shape[0]:,} rows x {feature_frame.shape[1]} columns\n")
print(feature_frame.head())

# --- Verification: PROVE the cutoff was actually respected, don't just assume it ---
verification = con.sql(f"""
    SELECT
        MIN(report_date) AS earliest_date_used,
        MAX(report_date) AS latest_date_used,
        MAX(report_date) <= DATE '{DECISION_POINT}' AS respects_decision_point
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}'
      AND report_date <= DATE '{DECISION_POINT}'
""").df()

print("\nVerification — did every row used actually respect the cutoff?")
print(verification.to_string(index=False))


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 176,738 rows x 7 columns

            client_hash_id           content_hash_id  total_impressions_30d  \
0  client_62f4a7e64f5e0096  content_39d7361b4945d504                   77.0   
1  client_62f4a7e64f5e0096  content_cec711b02f3bbde6                  602.0   
2  client_62f4a7e64f5e0096  content_275b6f7f733016d4                  810.0   
3  client_62f4a7e64f5e0096  content_ceaec531566ffcfc                   82.0   
4  client_62f4a7e64f5e0096  content_755d951187fcd70a                 1858.0   

   total_clicks_30d  ctr_30d  avg_position_30d  days_with_impressions_30d  
0               0.0   0.0000              4.07                         24  
1               4.0   0.0066              4.43                         29  
2               1.0   0.0012              4.87                         29  
3               0.0   0.0000              8.98                         27  
4               6.0   0.0032              1.85                         30  

Verification — did every ro

1. total_impressions_30d — knowable at the decision moment because it's a SUM over report_dates strictly ≤ 2026-03-31, verified above by MAX(report_date) <= DECISION_POINT.
2. total_clicks_30d — knowable at the decision moment because it uses the identical date-filtered rows as feature 1, so nothing after the cutoff contributes to it.
3. ctr_30d — knowable at the decision moment because it's a ratio of two already-knowable features (clicks ÷ impressions); combining two knowable quantities cannot introduce future information.
4. avg_position_30d — knowable at the decision moment because each day's position was recorded as of that day; averaging only pre-cutoff daily readings adds nothing from later.
5. days_with_impressions_30d — knowable at the decision moment because it's a COUNT(DISTINCT report_date) restricted to the same ≤-cutoff window as every other feature — counting only days that had already occurred.

### Adding leaky features & testing the precision

In [20]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# The trap: deliberately leak the label into a feature,
# watch the score jump toward perfect, then remove it.
# ============================================================
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score

DECISION_POINT = "2026-03-31"
LABEL_START    = "2026-04-01"   # strictly AFTER the decision point
LABEL_END      = "2026-05-01"   # exclusive upper bound (30-day label window)

# --- Build the LABEL from the genuinely later window (April) ---
label_frame = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_next30d
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{LABEL_START}' AND report_date < DATE '{LABEL_END}'
    GROUP BY client_hash_id, content_hash_id
""").df()

# --- Join March features (already built) to the April-based label ---
dataset = feature_frame.merge(label_frame, on=["client_hash_id", "content_hash_id"], how="inner")

dataset["pct_change"] = (
    (dataset["impressions_next30d"] - dataset["total_impressions_30d"])
    / dataset["total_impressions_30d"].replace(0, 1)
)
dataset["is_declining"] = (dataset["pct_change"] <= -0.20).astype(int)

print(f"Dataset: {len(dataset):,} rows | Decline rate: {dataset['is_declining'].mean():.3f}\n")

# ============================================================
# RUN 1 — HONEST features only (no leak)
# ============================================================
honest_features = ["total_impressions_30d", "total_clicks_30d", "ctr_30d",
                    "avg_position_30d", "days_with_impressions_30d"]

X = dataset[honest_features]
y = dataset["is_declining"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

model_honest = RandomForestClassifier(n_estimators=100, random_state=42)
model_honest.fit(X_train, y_train)
preds_honest = model_honest.predict_proba(X_test)[:, 1]

def precision_at_k(y_true, scores, k=50):
    order = scores.argsort()[::-1][:k]
    return y_true.values[order].mean()

p_honest = precision_at_k(y_test, preds_honest, k=50)
print(f"RUN 1 - Honest features only, Precision@50: {p_honest:.3f}\n")

# ============================================================
# RUN 2 — THE TRAP: inject a label-derived column on purpose
# ============================================================
# pct_change is LITERALLY the formula the label was thresholded from.
# This is the exact leakage move we already caught on the starter CSV
# (trend_pct -> trend_direction).
dataset["LEAKED_pct_change"] = dataset["pct_change"]

leaked_features = honest_features + ["LEAKED_pct_change"]
X_leak = dataset[leaked_features]
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_leak, y, test_size=0.2, random_state=42, stratify=y
)

model_leaked = RandomForestClassifier(n_estimators=100, random_state=42)
model_leaked.fit(X_train_l, y_train_l)
preds_leaked = model_leaked.predict_proba(X_test_l)[:, 1]

p_leaked = precision_at_k(y_test_l, preds_leaked, k=50)
print(f"RUN 2 - WITH the leaked column, Precision@50: {p_leaked:.3f}")
print(f"        (jumped by {p_leaked - p_honest:+.3f} vs. the honest run)\n")

# ============================================================
# RUN 3 — delete the leak, keep the honest number
# ============================================================
print("="*60)
print(f"FINAL HONEST RESULT (leak removed): Precision@50 = {p_honest:.3f}")
print(f"The leaked run's {p_leaked:.3f} is DISCARDED - it measured nothing")
print(f"real, only how well the model could read its own answer key.")
print("="*60)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Dataset: 176,737 rows | Decline rate: 0.534

RUN 1 - Honest features only, Precision@50: 0.620

RUN 2 - WITH the leaked column, Precision@50: 1.000
        (jumped by +0.380 vs. the honest run)

FINAL HONEST RESULT (leak removed): Precision@50 = 0.620
The leaked run's 1.000 is DISCARDED - it measured nothing
real, only how well the model could read its own answer key.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
One limitation of my slice:

**Unbalanced history** — different clients have wildly different amounts of data

**What it means:** the guide states the daily data starts 2025-01-27 at the earliest, but each client's own tracking began whenever they actually started using GSC/GA4 — so one client might have 17 months of history while another has 6 weeks. Only 9 of 70 clients have 12+ months, which the guide flags as the minimum needed for seasonality work.

**Why it matters for your model:** if you train on all clients pooled together, clients with long histories dominate the training signal, and any pattern you find might just reflect those few data-rich clients rather than being universal.

In [21]:
# ============================================================
# Section 4 — Data limits, verified with real queries
# ============================================================

print("="*70)
print("LIMIT 1 — Unbalanced history across clients")
print("="*70)
history_check = con.sql(f"""
    SELECT
        client_hash_id,
        gsc_data_start,
        ga4_data_start,
        DATE_DIFF('day', gsc_data_start, DATE '2026-06-30') AS days_of_gsc_history
    FROM {TABLES['dim_clients']}
    ORDER BY days_of_gsc_history DESC
""").df()

print(f"Shortest history: {history_check['days_of_gsc_history'].min()} days")
print(f"Longest history:  {history_check['days_of_gsc_history'].max()} days")
print(f"Median history:   {history_check['days_of_gsc_history'].median():.0f} days")
n_12mo_plus = (history_check["days_of_gsc_history"] >= 365).sum()
print(f"Clients with 12+ months (>=365 days) of history: {n_12mo_plus} of {len(history_check)}\n")

print("="*70)
print("LIMIT 2 — GSC-only rows: how many March rows predate a client's GA4 start?")
print("="*70)
gsc_only_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_march_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS rows_with_ga4,
        SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS rows_gsc_only,
        SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) AS rows_unknown_status
    FROM {TABLES['fact_daily']}
    WHERE report_date >= DATE '{MONTH_START}' AND report_date <= DATE '{DECISION_POINT}'
""").df()
print(gsc_only_check.to_string(index=False))
pct_gsc_only = gsc_only_check['rows_gsc_only'][0] / gsc_only_check['total_march_rows'][0] * 100
print(f"\n-> {pct_gsc_only:.1f}% of March rows are GSC-only (no engagement data available yet).\n")

print("="*70)
print("LIMIT 3 — Window overlap check: do feature and label windows share any dates?")
print("="*70)
overlap_check = con.sql(f"""
    SELECT
        (SELECT MAX(report_date) FROM {TABLES['fact_daily']}
         WHERE report_date <= DATE '{DECISION_POINT}') AS last_feature_date,
        (SELECT MIN(report_date) FROM {TABLES['fact_daily']}
         WHERE report_date >= DATE '{LABEL_START}') AS first_label_date
""").df()
print(overlap_check.to_string(index=False))
no_overlap = overlap_check['last_feature_date'][0] < overlap_check['first_label_date'][0]
print(f"\nFeature window strictly ends before label window begins: {no_overlap}")

LIMIT 1 — Unbalanced history across clients
Shortest history: 28 days
Longest history:  519 days
Median history:   237 days
Clients with 12+ months (>=365 days) of history: 9 of 104

LIMIT 2 — GSC-only rows: how many March rows predate a client's GA4 start?
 total_march_rows  rows_with_ga4  rows_gsc_only  rows_unknown_status
          9841378       413966.0      6408671.0            3018741.0

-> 65.1% of March rows are GSC-only (no engagement data available yet).

LIMIT 3 — Window overlap check: do feature and label windows share any dates?


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

last_feature_date first_label_date
       2026-03-31       2026-04-01

Feature window strictly ends before label window begins: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.